In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{F - Features} $$
$$ \text{C - Classes} $$
$$
y = \begin{pmatrix}
y_{0} & y_{1} & \cdots & y_{C} \\
\end{pmatrix}
=
\begin{pmatrix}
x_{0} & x_{1} & \cdots & x_{F} \\
\end{pmatrix} 
\cdot
\begin{pmatrix}
w_{00} & w_{01} & \cdots & w_{0C} \\
w_{10} & w_{11} & \cdots & w_{1C} \\
\vdots & \vdots & \ddots & \vdots \\
w_{F0} & w_{F1} & \cdots & w_{FC} \\
\end{pmatrix}
+
\begin{pmatrix}
b_0 & b_1 & \cdots & b_C \\
\end{pmatrix}
$$

$$ probabilities = softmax(y) = softmax(y_0, y_1, \dots, y_C) $$

In [ ]:
from torch import float32, tensor
from torch.nn import Linear
from torch.nn.functional import softmax, cross_entropy
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore


def logistic_regression_nc_sgd_autograd(X, y, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    ((s, f), c) = (X.shape, len(set(y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))
    assert_eq(y.shape, (s, 1))

    model = Linear(f, c)
    optimizer = SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        logits = model(X)
        assert_eq(logits.shape, (s, c))

        loss = cross_entropy(logits, y.squeeze().long())
        assert_eq(loss.shape, ())
        
        loss.backward()
        optimizer.step()

    return (loss.item(), model)


def _test_logistic_regression_nc_sgd_autograd(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of three classes: healthy (0), viral (1), and bacterial (2). 
    The model is trained on 70 samples and then evaluated on 30 samples (10 per class). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set.
    """

    training_data = T([Patient([0.5, 0.25, 0.25]).data for _ in range(70)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling   

    y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_nc_sgd_autograd(X, y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        x = T(d[:-1])
        y = T(d[-1])

        x[0] /= 100
        x[5] /= 200
        
        total += 1
        correct += check_eq(softmax(model(x), dim=0).argmax(), y)

    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_autograd()